# PageIndex RAG — Cloud API Step-by-Step Guide

> **No Vector DB. No Chunking. No Similarity Search.** PageIndex uses LLM reasoning over a hierarchical tree index to retrieve exactly the right pages — the way a human expert would.

## What You'll Learn

1. **Why vectorless RAG?** — The problem with similarity search
2. **Setup** — Install PageIndex SDK, configure API keys
3. **Index a PDF** — Submit a document to PageIndex Cloud
4. **Explore the Tree** — View the hierarchical structure
5. **Reasoning-Based Retrieval** — LLM tree search to find relevant nodes
6. **Answer Generation** — Generate grounded answers from retrieved context
7. **Chat API** — Use PageIndex's built-in chat for instant Q&A

## How PageIndex Works

```
                    Traditional Vector RAG
PDF → Chunks → Embeddings → Vector DB → Cosine Similarity → Top-K → LLM → Answer
                    ❌ similarity ≠ relevance

                    PageIndex RAG (this notebook)
PDF → Tree Index → LLM Reasons Over Tree → Fetch Exact Pages → LLM → Answer
                    ✅ reasoning = relevance
```

| Feature | Vector RAG | PageIndex RAG |
|---------|-----------|---------------|
| Retrieval method | Cosine similarity | LLM reasoning |
| Chunking required | Yes | No |
| Vector DB required | Yes | No |
| Explainability | Low (black box) | High (traceable) |
| Multi-hop reasoning | Hard | Native |
| FinanceBench accuracy | ~80% | **98.7%** |

## Prerequisites

- Python 3.9+
- An [OpenAI API key](https://platform.openai.com/api-keys) (for tree search & answer generation)
- A [PageIndex API key](https://dash.pageindex.ai/api-keys) (free tier available)

---

**Part of [Awesome GenAI Toolkit](https://github.com/shubh-vedi/awesome-genai-toolkit)** — Star the repo if this helps you!

## Step 0: Install Dependencies

In [ ]:
!pip install -q --upgrade pageindex openai

## Step 1: Setup & Configuration

You need two API keys:
- **PageIndex API key** — for document indexing and tree generation ([get one here](https://dash.pageindex.ai/api-keys))
- **OpenAI API key** — for LLM reasoning during tree search and answer generation

In [ ]:
import os
import json
import requests
import openai
from pprint import pprint

# --- API Key Configuration ---
# Option 1: Set your keys directly (for quick testing)
# os.environ["OPENAI_API_KEY"] = "sk-..."
# PAGEINDEX_API_KEY = "your-pageindex-api-key"

# Option 2: Google Colab secrets (recommended for Colab)
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    PAGEINDEX_API_KEY = userdata.get("PAGEINDEX_API_KEY")
except ImportError:
    PAGEINDEX_API_KEY = os.environ.get("PAGEINDEX_API_KEY", "YOUR_PAGEINDEX_API_KEY")

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
LLM_MODEL = "gpt-4o-mini"  # Used for tree search reasoning & answer generation

print(f"OpenAI key configured: {'Yes' if OPENAI_API_KEY else 'No — set OPENAI_API_KEY!'}")
print(f"PageIndex key configured: {'Yes' if PAGEINDEX_API_KEY != 'YOUR_PAGEINDEX_API_KEY' else 'No — set PAGEINDEX_API_KEY!'}")
print(f"LLM Model: {LLM_MODEL}")

In [ ]:
from pageindex import PageIndexClient
import pageindex.utils as utils

# --- Initialize the PageIndex Cloud client ---
pi_client = PageIndexClient(api_key=PAGEINDEX_API_KEY)

print("PageIndex Cloud client initialized.")

In [ ]:
# --- Helper: async LLM call for tree search & answer generation ---
async def call_llm(prompt: str, model: str = LLM_MODEL, temperature: float = 0) -> str:
    """Call OpenAI chat completion and return the response text."""
    client = openai.AsyncOpenAI(api_key=OPENAI_API_KEY)
    response = await client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature,
    )
    return response.choices[0].message.content.strip()

print("LLM helper ready.")

## Step 2: Download & Submit a PDF

We'll download an arXiv paper and submit it to the PageIndex Cloud API for indexing. The cloud service builds the hierarchical tree structure for you — no local LLM calls needed.

You can replace this with any PDF of your choice.

In [ ]:
# --- Download a sample PDF (DeepSeek-R1 paper) ---
PDF_URL = "https://arxiv.org/pdf/2501.12948.pdf"
PDF_DIR = "data"
PDF_PATH = os.path.join(PDF_DIR, "deepseek_r1.pdf")

os.makedirs(PDF_DIR, exist_ok=True)

if not os.path.exists(PDF_PATH):
    print(f"Downloading paper from {PDF_URL} ...")
    resp = requests.get(PDF_URL, timeout=60)
    resp.raise_for_status()
    with open(PDF_PATH, "wb") as f:
        f.write(resp.content)
    print(f"Saved to {PDF_PATH} ({len(resp.content) / 1024:.0f} KB)")
else:
    print(f"PDF already exists: {PDF_PATH}")

print(f"File size: {os.path.getsize(PDF_PATH) / 1024:.0f} KB")

In [ ]:
# --- Submit the PDF to PageIndex Cloud for indexing ---
result = pi_client.submit_document(PDF_PATH)
doc_id = result["doc_id"]
print(f"Document submitted successfully!")
print(f"Document ID: {doc_id}")

### 2.1 Check Processing Status

The cloud API processes the document asynchronously. For large PDFs this may take a minute or two. Re-run this cell until the status shows `completed`.

In [ ]:
# --- Check if the document is ready ---
import time

doc_info = pi_client.get_document(doc_id)
pprint(doc_info)

if doc_info["status"] == "completed":
    print(f"\nDocument ready! ({doc_info['pageNum']} pages)")
elif doc_info["status"] == "processing":
    print("\nDocument is still processing. Wait a moment and re-run this cell.")
    print("Tip: For long documents, this may take 1-2 minutes.")

## Step 3: Explore the Tree Structure

Once processing is complete, PageIndex has built a **hierarchical tree index** of your document. This is the key innovation — instead of flat chunks, you get a structured representation:

```
Your PDF
   ↓
[PageIndex Tree]
├── Section 1 (pages 1-5)    ← LLM reasons: "relevant to query?"
│   ├── Sub-section 1.1
│   └── Sub-section 1.2
├── Section 2 (pages 6-12)   ← LLM reasons: "not relevant, skip"
└── Section 3 (pages 13-20)  ← LLM reasons: "highly relevant!"
```

Each node has a **title**, **page range**, and **summary** — no embeddings anywhere!

In [ ]:
# --- Get and display the tree structure ---
if pi_client.is_retrieval_ready(doc_id):
    tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
    print("Document Tree Structure:")
    print("=" * 60)
    utils.print_tree(tree)
else:
    print("Document not ready yet. Re-run the status check cell above.")

In [ ]:
# --- Inspect the raw tree JSON ---
# Each node contains: title, node_id, start_index, end_index, summary, and child nodes
tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])
print("Sample tree node (JSON):")
print(json.dumps(tree_without_text, indent=2)[:2000])
print("...")

## Step 4: Reasoning-Based Retrieval (Tree Search)

This is where PageIndex shines. Instead of cosine similarity, we ask the LLM to **reason** over the tree structure and identify which nodes are relevant to our query.

The LLM sees the tree (titles + summaries) and decides which sections to retrieve — just like a human expert skimming a table of contents.

### How it works:
1. We strip the text from the tree (LLM only sees structure + summaries)
2. We ask the LLM: "Which nodes likely contain the answer?"
3. The LLM reasons through the tree and returns a list of node IDs
4. We fetch the full text only from those nodes

In [ ]:
# --- Define the query ---
query = "What are the main conclusions and results of this paper?"
print(f"Query: {query}")

In [ ]:
# --- LLM reasons over the tree to find relevant nodes ---

# Remove text content — LLM only sees structure + summaries (lightweight)
tree_without_text = utils.remove_fields(tree.copy(), fields=["text"])

TREE_SEARCH_PROMPT = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Your task is to find all nodes that are likely to contain the answer to the question.

Question: {query}

Document tree structure:
{json.dumps(tree_without_text, indent=2)}

Please reply in the following JSON format:
{{
    "thinking": "<Your thinking process on which nodes are relevant to the question>",
    "node_list": ["node_id_1", "node_id_2", ..., "node_id_n"]
}}
Directly return the final JSON structure. Do not output anything else.
"""

print("Asking LLM to reason over the tree structure...")
tree_search_result = await call_llm(TREE_SEARCH_PROMPT)
tree_search_json = json.loads(tree_search_result)
print("Done! LLM identified relevant nodes.")

In [ ]:
# --- View the LLM's reasoning process ---
print("LLM Reasoning Process:")
print("=" * 60)
utils.print_wrapped(tree_search_json["thinking"])

# Build a node lookup map for easy access
node_map = utils.create_node_mapping(tree)

print("\n\nRetrieved Nodes:")
print("-" * 60)
for node_id in tree_search_json["node_list"]:
    node = node_map[node_id]
    page_info = node.get("page_index", f"{node.get('start_index', '?')}-{node.get('end_index', '?')}")
    print(f"  Node {node['node_id']}  |  Pages: {page_info}  |  {node['title']}")

## Step 5: Extract Context & Generate Answer

Now we fetch the actual text content from the retrieved nodes and pass it to the LLM for grounded answer generation.

In [ ]:
# --- Extract text from retrieved nodes ---
node_list = tree_search_json["node_list"]
relevant_content = "\n\n".join(node_map[nid]["text"] for nid in node_list)

print(f"Extracted {len(relevant_content):,} characters from {len(node_list)} nodes")
print(f"\nPreview of retrieved context:")
print("=" * 60)
utils.print_wrapped(relevant_content[:1500] + "\n...")

In [ ]:
# --- Generate answer from retrieved context ---
ANSWER_PROMPT = f"""
Answer the question based on the context below.

Rules:
- Answer ONLY from the provided context. Do not use prior knowledge.
- If the context doesn't contain the answer, say "I don't have enough information."
- Reference which section/pages your answer comes from.
- Be concise and well-structured.

Context:
{relevant_content}

Question: {query}
"""

print(f"Question: {query}")
print("=" * 60)
answer = await call_llm(ANSWER_PROMPT)
print(f"\nAnswer:\n")
utils.print_wrapped(answer)

## Step 6: Reusable PageIndex RAG Pipeline

Let's wrap everything into a clean, reusable function and try multiple queries.

In [ ]:
async def pageindex_rag(query: str, tree: dict, node_map: dict, verbose: bool = True) -> str:
    """
    Complete PageIndex RAG pipeline:
    1. LLM reasons over tree structure to find relevant nodes
    2. Extract text from those nodes
    3. Generate answer from the retrieved context
    """
    # Step 1: Tree search — LLM identifies relevant nodes
    tree_no_text = utils.remove_fields(tree.copy(), fields=["text"])
    search_prompt = f"""
You are given a question and a tree structure of a document.
Each node contains a node id, node title, and a corresponding summary.
Find all nodes likely to contain the answer.

Question: {query}

Document tree structure:
{json.dumps(tree_no_text, indent=2)}

Reply in JSON format:
{{
    "thinking": "<reasoning>",
    "node_list": ["node_id_1", "node_id_2"]
}}
Return only the JSON.
"""
    search_result = json.loads(await call_llm(search_prompt))

    if verbose:
        print(f"  Reasoning: {search_result['thinking'][:200]}...")
        print(f"  Retrieved {len(search_result['node_list'])} nodes")

    # Step 2: Extract context from relevant nodes
    context = "\n\n".join(node_map[nid]["text"] for nid in search_result["node_list"])

    # Step 3: Generate grounded answer
    answer_prompt = f"""Answer the question based only on this context. Be concise and cite sections.

Context: {context}

Question: {query}"""

    return await call_llm(answer_prompt)

print("pageindex_rag() function ready!")

In [ ]:
# --- Try multiple questions ---
questions = [
    "What training methodology was used in this paper?",
    "What benchmarks were evaluated, and what were the key results?",
    "What are the limitations discussed by the authors?",
]

for q in questions:
    print(f"Q: {q}")
    ans = await pageindex_rag(q, tree, node_map)
    print(f"\nA: {ans}\n")
    print("-" * 80)

## Step 7: PageIndex Chat API (Shortcut)

PageIndex also provides a **built-in Chat API** that handles retrieval + answer generation in one call. This is the fastest way to get answers — no manual tree search needed.

In [ ]:
# --- Use the built-in Chat API (streaming) ---
chat_query = "Explain the key contributions of this paper in simple terms."
print(f"Query: {chat_query}\n")
print("Answer:")
print("=" * 60)

for chunk in pi_client.chat_completions(
    messages=[{"role": "user", "content": chat_query}],
    doc_id=doc_id,
    stream=True,
):
    print(chunk, end="", flush=True)

print()  # newline after streaming

In [ ]:
# --- Try more questions with the Chat API ---
chat_questions = [
    "What model architecture is described?",
    "How does this compare to GPT-4? Show me the page numbers.",
    "Summarize the abstract in 3 bullet points.",
]

for q in chat_questions:
    print(f"\nQ: {q}")
    print("-" * 40)
    for chunk in pi_client.chat_completions(
        messages=[{"role": "user", "content": q}],
        doc_id=doc_id,
        stream=True,
    ):
        print(chunk, end="", flush=True)
    print("\n")

## Step 8: Try With Your Own PDF

Replace the URL below with your own document and re-run from Step 2!

In [ ]:
# --- Upload your own PDF ---
# Option 1: Upload in Colab
# from google.colab import files
# uploaded = files.upload()
# my_pdf_path = list(uploaded.keys())[0]

# Option 2: Download from URL
# my_pdf_url = "https://example.com/your-document.pdf"
# my_pdf_path = "data/my_document.pdf"
# resp = requests.get(my_pdf_url, timeout=60)
# with open(my_pdf_path, "wb") as f:
#     f.write(resp.content)

# Then submit and query:
# my_doc_id = pi_client.submit_document(my_pdf_path)["doc_id"]
# # ... wait for processing, then query as shown above

## Recap: The Full PageIndex Cloud RAG Pipeline

```
1. pip install pageindex
2. pi_client = PageIndexClient(api_key="...")
3. doc_id = pi_client.submit_document("paper.pdf")["doc_id"]
4. tree = pi_client.get_tree(doc_id, node_summary=True)["result"]
5. LLM reasons over tree → identifies relevant nodes
6. Fetch text from those nodes → generate answer
   — OR —
   pi_client.chat_completions(messages=[...], doc_id=doc_id)  # one-liner
```

## What's Next?

| Notebook | Description |
|----------|-------------|
| **PageIndex Local (Self-Hosted)** | Run PageIndex on your own machine — coming soon |
| **Agentic RAG with PageIndex** | LLM agent autonomously navigates the tree — coming soon |

---

### Resources

- [PageIndex GitHub](https://github.com/VectifyAI/PageIndex) — Star the repo!
- [PageIndex Docs](https://docs.pageindex.ai) — API reference
- [PageIndex Dashboard](https://dash.pageindex.ai) — Manage documents & API keys
- [Discord](https://discord.com/invite/VuXuf29EUj) — Community support

---

### Found this useful?

**[Star the Awesome GenAI Toolkit repo](https://github.com/shubh-vedi/awesome-genai-toolkit)** — it helps others discover these tutorials!

More notebooks, apps, and guides are being added every week. Contributions welcome!